# Phase 1 — Data Audit & Exploratory Data Analysis
### Deep Learning-Based Predictive Maintenance and Remaining Useful Life (RUL) Estimation  ·  TensorFlow / Keras

**Goal of this phase:** understand the data *before* modelling. We (1) obtain the NASA C-MAPSS turbofan dataset, (2) audit its quality,
(3) explore how sensors behave as an engine degrades, and (4) turn what we see into **concrete preprocessing decisions** for Phase 2.

**Run order:** Phase 1 → Phase 2 → Phase 3. This notebook is *self-contained*: it writes the small helper library `src/rul_core.py` itself.

| Term | Meaning |
|---|---|
| Engine / unit | one machine followed from new until failure (training) or until a random cut-off (test) |
| Cycle | one flight = one row of 21 sensor readings + 3 operating settings |
| RUL | Remaining Useful Life = cycles left before failure |
| FD001…FD004 | four sub-datasets (1 or 6 operating conditions, 1 or 2 fault modes) |

## 0. Shared helper library
This cell writes `src/rul_core.py` (data loading, preprocessing, windowing, metrics). It is plain NumPy / pandas / scikit-learn — no deep-learning framework is needed in Phase 1 or 2.

In [ ]:
%%writefile src/rul_core.py
"""
rul_core.py  -  data loading, preprocessing, windowing, metrics, anomaly detection.

Pure NumPy / pandas / scikit-learn (NO TensorFlow here), so every step of Phase 1
and Phase 2 can be run, tested and explained without a deep-learning framework.

Dataset : NASA C-MAPSS turbofan engine degradation (FD001 ... FD004).
Task    : given the last `window` cycles of sensor readings, predict Remaining Useful Life (RUL).
"""
from __future__ import annotations

import io
import json
import re
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
RUL_CAP = 125                      # piece-wise linear RUL target (standard practice for C-MAPSS)
ID_COLS = ["unit_id", "cycle"]
SETTING_COLS = [f"setting_{i}" for i in range(1, 4)]
SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]
ALL_COLS = ID_COLS + SETTING_COLS + SENSOR_COLS

# Per-dataset configuration. FD002/FD004 have 6 operating conditions -> need per-condition normalisation.
DATASET_CFG = {
    "FD001": dict(n_conditions=1, window=30, train_stride=1, use_cycle=False, description="1 operating condition, 1 fault mode (HPC)"),
    "FD002": dict(n_conditions=6, window=40, train_stride=2, use_cycle=True, description="6 operating conditions, 1 fault mode"),
    "FD003": dict(n_conditions=1, window=30, train_stride=1, use_cycle=False, description="1 operating condition, 2 fault modes"),
    "FD004": dict(n_conditions=6, window=40, train_stride=2, use_cycle=True, description="6 operating conditions, 2 fault modes (hardest)"),
}

SENSOR_INFO = {  # official C-MAPSS sensor descriptions
    "sensor_1": ("T2", "Total temperature at fan inlet"), "sensor_2": ("T24", "Total temperature at LPC outlet"),
    "sensor_3": ("T30", "Total temperature at HPC outlet"), "sensor_4": ("T50", "Total temperature at LPT outlet"),
    "sensor_5": ("P2", "Pressure at fan inlet"), "sensor_6": ("P15", "Total pressure in bypass duct"),
    "sensor_7": ("P30", "Total pressure at HPC outlet"), "sensor_8": ("Nf", "Physical fan speed"),
    "sensor_9": ("Nc", "Physical core speed"), "sensor_10": ("epr", "Engine pressure ratio"),
    "sensor_11": ("Ps30", "Static pressure at HPC outlet"), "sensor_12": ("phi", "Ratio of fuel flow to Ps30"),
    "sensor_13": ("NRf", "Corrected fan speed"), "sensor_14": ("NRc", "Corrected core speed"),
    "sensor_15": ("BPR", "Bypass ratio"), "sensor_16": ("farB", "Burner fuel-air ratio"),
    "sensor_17": ("htBleed", "Bleed enthalpy"), "sensor_18": ("Nf_dmd", "Demanded fan speed"),
    "sensor_19": ("PCNfR_dmd", "Demanded corrected fan speed"), "sensor_20": ("W31", "HPT coolant bleed"),
    "sensor_21": ("W32", "LPT coolant bleed"),
}
# Groups used by the dashboard (C-MAPSS has no vibration sensor; fan/core speed is the rotating-machinery analogue)
SENSOR_GROUPS = {
    "Temperature": ["sensor_2", "sensor_3", "sensor_4"],
    "Pressure": ["sensor_7", "sensor_11", "sensor_6"],
    "Speed / Vibration proxy": ["sensor_8", "sensor_9", "sensor_13", "sensor_14"],
    "Flow / Fuel": ["sensor_12", "sensor_15", "sensor_20", "sensor_21"],
}

DATA_URLS = [
    "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
]
_FILE_RE = re.compile(r"^(train|test|RUL)_FD00[1-4]\.txt$")


# ----------------------------------------------------------------------------------------------
# 1. Data acquisition and loading
# ----------------------------------------------------------------------------------------------
def find_cmapss_files(root) -> dict:
    """Return {filename: path} for every C-MAPSS text file found below `root`."""
    root = Path(root)
    return {p.name: p for p in root.rglob("*.txt") if _FILE_RE.match(p.name)} if root.exists() else {}


def _extract_recursive(zip_bytes: bytes, dest: Path, depth: int = 0) -> None:
    """Extract a zip; if it contains further zips (the NASA mirror does), extract those too."""
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for name in zf.namelist():
            data = zf.read(name)
            if name.lower().endswith(".zip") and depth < 3:
                _extract_recursive(data, dest, depth + 1)
            elif name.lower().endswith(".txt"):
                (dest / Path(name).name).write_bytes(data)


def ensure_dataset(data_dir, datasets=("FD001", "FD004"), verbose=True) -> dict:
    """
    Make sure the C-MAPSS text files exist in `data_dir`.
    Order: (1) already present -> done, (2) a CMAPSSData*.zip placed in data_dir, (3) download from NASA mirrors.
    Raises a FileNotFoundError with plain instructions if everything fails.
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    need = [f"{p}_{d}.txt" for d in datasets for p in ("train", "test", "RUL")]

    def complete():
        found = find_cmapss_files(data_dir)
        return all(n in found for n in need), found

    ok, found = complete()
    if ok:
        if verbose:
            print(f"Dataset already available in {data_dir}")
        return found

    for z in sorted(data_dir.rglob("*.zip")):                       # (2) user-supplied zip
        try:
            _extract_recursive(z.read_bytes(), data_dir)
            ok, found = complete()
            if ok:
                if verbose:
                    print(f"Extracted dataset from {z.name}")
                return found
        except zipfile.BadZipFile:
            pass

    try:                                                            # (3) download
        import requests
    except ImportError:
        requests = None
    if requests is not None:
        for url in DATA_URLS:
            try:
                if verbose:
                    print("Trying", url)
                r = requests.get(url, timeout=(15, 180))
                r.raise_for_status()
                _extract_recursive(r.content, data_dir)
                ok, found = complete()
                if ok:
                    if verbose:
                        print("Download OK")
                    return found
            except Exception as exc:                                # network problems are expected sometimes
                if verbose:
                    print(f"  failed: {type(exc).__name__}: {str(exc)[:100]}")
    raise FileNotFoundError(
        "C-MAPSS files not found and automatic download failed.\n"
        "FIX (30 seconds): download 'CMAPSSData.zip' from NASA (search: 'NASA C-MAPSS turbofan dataset'),\n"
        f"and put the ZIP (or the extracted .txt files) inside:  {data_dir.resolve()}\n"
        "Then re-run this cell. In Google Colab: use the Files panel (left) -> drag the ZIP into data/raw."
    )


def read_table(path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None).dropna(axis=1, how="all")
    if df.shape[1] != len(ALL_COLS):
        raise ValueError(f"{Path(path).name}: expected {len(ALL_COLS)} columns, found {df.shape[1]}")
    df.columns = ALL_COLS
    df["unit_id"] = df["unit_id"].astype(int)
    df["cycle"] = df["cycle"].astype(int)
    return df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)


def load_subset(files: dict, code: str):
    """Return (train_df, test_df, test_rul_array) for 'FD001'..'FD004'."""
    train = read_table(files[f"train_{code}.txt"])
    test = read_table(files[f"test_{code}.txt"])
    rul = pd.read_csv(files[f"RUL_{code}.txt"], sep=r"\s+", header=None).dropna(axis=1, how="all").iloc[:, 0].to_numpy(float)
    if len(rul) != test["unit_id"].nunique():
        raise ValueError(f"{code}: {len(rul)} RUL labels but {test['unit_id'].nunique()} test engines")
    return train, test, rul


def add_rul(train: pd.DataFrame, cap: float = RUL_CAP) -> pd.DataFrame:
    """RUL = (last cycle of the engine) - (current cycle); capped at `cap` (early life = 'healthy plateau')."""
    out = train.copy()
    out["RUL_raw"] = out.groupby("unit_id")["cycle"].transform("max") - out["cycle"]
    out["RUL"] = out["RUL_raw"].clip(upper=cap)
    return out


def split_engines(unit_ids, val_fraction=0.2, seed=SEED):
    """Split by ENGINE (never by row) so no engine is in both train and validation -> no leakage."""
    rng = np.random.RandomState(seed)
    ids = np.array(sorted(unit_ids))
    rng.shuffle(ids)
    n_val = max(1, int(round(len(ids) * val_fraction)))
    return np.sort(ids[n_val:]), np.sort(ids[:n_val])


# ----------------------------------------------------------------------------------------------
# 2. Preprocessing  (fit on TRAIN engines only)
# ----------------------------------------------------------------------------------------------
class Preprocessor:
    """
    Steps (all parameters are learnt on training engines only and stored as plain numbers -> JSON):
      1. operating-condition detection (k-means on the 3 settings) for multi-condition sets (FD002/FD004)
      2. per-condition z-score of every sensor   (removes the effect of altitude / speed / throttle)
      3. drop sensors that are constant (carry no information)
      4. causal exponential smoothing per engine (denoises without looking into the future)
      5. optional 'cycle' feature (operating hours so far - always known at prediction time)
    """

    def __init__(self, n_conditions=1, smooth_span=5, use_cycle=True, cap=RUL_CAP, window=30):
        self.n_conditions, self.smooth_span, self.use_cycle = n_conditions, smooth_span, use_cycle
        self.cap, self.window = cap, window

    # -- operating conditions -------------------------------------------------
    def _cond(self, df):
        if self.n_conditions == 1:
            return np.zeros(len(df), dtype=int)
        s = (df[SETTING_COLS].to_numpy(float) - self.set_mu) / self.set_sd
        d = ((s[:, None, :] - self.centers[None, :, :]) ** 2).sum(-1)
        return d.argmin(1)

    def fit(self, train_df: pd.DataFrame):
        if self.n_conditions > 1:
            from sklearn.cluster import KMeans
            S = train_df[SETTING_COLS].to_numpy(float)
            self.set_mu, self.set_sd = S.mean(0), S.std(0) + 1e-8
            km = KMeans(self.n_conditions, n_init=10, random_state=SEED).fit((S - self.set_mu) / self.set_sd)
            self.centers = km.cluster_centers_
        cond = self._cond(train_df)
        X = train_df[SENSOR_COLS].to_numpy(float)
        C = self.n_conditions
        mu, sd = np.zeros((C, X.shape[1])), np.ones((C, X.shape[1]))
        for c in range(C):
            m = cond == c
            if m.sum() > 1:
                mu[c], sd[c] = X[m].mean(0), X[m].std(0)
        keep = sd.max(0) > 1e-6
        self.sensors = [s for s, k in zip(SENSOR_COLS, keep) if k]
        self._keep = keep
        self.mu, self.sd = mu, np.where(sd < 1e-6, 1.0, sd)
        self.cycle_scale = float(train_df["cycle"].max())
        return self

    @property
    def features(self):
        return self.sensors + (["cycle"] if self.use_cycle else [])

    @property
    def n_features(self):
        return len(self.features)

    # -- transform --------------------------------------------------------------
    def transform_engine(self, df: pd.DataFrame) -> np.ndarray:
        """df = rows of ONE engine sorted by cycle -> (n_cycles, n_features) float32."""
        cond = self._cond(df)
        X = df[SENSOR_COLS].to_numpy(float)
        Z = ((X - self.mu[cond]) / self.sd[cond])[:, self._keep]
        if self.smooth_span > 1:
            Z = pd.DataFrame(Z).ewm(span=self.smooth_span, adjust=False).mean().to_numpy()
        if self.use_cycle:
            Z = np.column_stack([Z, df["cycle"].to_numpy(float) / self.cycle_scale])
        return Z.astype(np.float32)

    def sensor_part(self, feat: np.ndarray) -> np.ndarray:
        return feat[:, : len(self.sensors)]

    # -- (de)serialisation -----------------------------------------------------
    def to_dict(self):
        d = dict(n_conditions=self.n_conditions, smooth_span=self.smooth_span, use_cycle=self.use_cycle, cap=self.cap,
                 window=self.window, sensors=self.sensors, keep=self._keep.tolist(), mu=self.mu.tolist(),
                 sd=self.sd.tolist(), cycle_scale=self.cycle_scale)
        if self.n_conditions > 1:
            d.update(set_mu=self.set_mu.tolist(), set_sd=self.set_sd.tolist(), centers=self.centers.tolist())
        return d

    @classmethod
    def from_dict(cls, d):
        p = cls(d["n_conditions"], d["smooth_span"], d["use_cycle"], d["cap"], d["window"])
        p.sensors, p._keep = d["sensors"], np.array(d["keep"], dtype=bool)
        p.mu, p.sd, p.cycle_scale = np.array(d["mu"]), np.array(d["sd"]), d["cycle_scale"]
        if p.n_conditions > 1:
            p.set_mu, p.set_sd, p.centers = np.array(d["set_mu"]), np.array(d["set_sd"]), np.array(d["centers"])
        return p


# ----------------------------------------------------------------------------------------------
# 3. Sliding windows
# ----------------------------------------------------------------------------------------------
def pad_front(feat: np.ndarray, window: int) -> np.ndarray:
    """Repeat the first row so that every cycle (even cycle 1) has a full window of history."""
    return np.vstack([np.repeat(feat[:1], window - 1, axis=0), feat])


def engine_windows(feat: np.ndarray, window: int) -> np.ndarray:
    """All windows of one engine: window i ends at cycle i  -> (n_cycles, window, F)."""
    padded = pad_front(feat, window)
    w = np.lib.stride_tricks.sliding_window_view(padded, window, axis=0)   # (n, F, window)
    return np.ascontiguousarray(w.transpose(0, 2, 1))


def build_windows(prep: Preprocessor, df: pd.DataFrame, window: int, stride: int = 1):
    """Training/validation windows for labelled engines. Returns X, y(capped), y_raw, unit, cycle."""
    X, y, yr, u, c = [], [], [], [], []
    for uid, g in df.groupby("unit_id", sort=True):
        g = g.sort_values("cycle")
        W = engine_windows(prep.transform_engine(g), window)
        idx = np.arange(len(g) - 1, -1, -stride)[::-1]              # always keep the last cycle of the engine
        X.append(W[idx]); y.append(g["RUL"].to_numpy()[idx]); yr.append(g["RUL_raw"].to_numpy()[idx])
        u.append(np.full(len(idx), uid)); c.append(g["cycle"].to_numpy()[idx])
    return (np.concatenate(X).astype(np.float32), np.concatenate(y).astype(np.float32),
            np.concatenate(yr).astype(np.float32), np.concatenate(u), np.concatenate(c))


def build_test_windows(prep: Preprocessor, test_df: pd.DataFrame, rul_true: np.ndarray, window: int):
    """One window per test engine: its LAST `window` observed cycles (this is the official C-MAPSS protocol)."""
    X, units, last = [], [], []
    for uid, g in test_df.groupby("unit_id", sort=True):
        g = g.sort_values("cycle")
        X.append(engine_windows(prep.transform_engine(g), window)[-1]); units.append(uid); last.append(int(g["cycle"].max()))
    return np.stack(X).astype(np.float32), np.asarray(rul_true, dtype=np.float32), np.array(units), np.array(last)


# ----------------------------------------------------------------------------------------------
# 4. Metrics
# ----------------------------------------------------------------------------------------------
def nasa_score(y_true, y_pred):
    """Asymmetric PHM08 score: late predictions (pred > true) are penalised more. Lower is better."""
    d = np.asarray(y_pred, float) - np.asarray(y_true, float)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)))


def regression_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    err = y_pred - y_true
    ss_res, ss_tot = float(np.sum(err ** 2)), float(np.sum((y_true - y_true.mean()) ** 2))
    return {"MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))),
            "R2": 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan"),
            "NASA_score": nasa_score(y_true, y_pred), "Within_10_cycles_%": float(np.mean(np.abs(err) <= 10) * 100)}


def classification_metrics(y_true, y_pred, y_score=None) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"Precision": float(p), "Recall": float(r), "F1": float(f)}
    if y_score is not None and len(np.unique(y_true)) == 2:
        out["ROC_AUC"] = float(roc_auc_score(y_true, y_score))
    return out


# ----------------------------------------------------------------------------------------------
# 5. Classical (non-deep) baselines: hand-crafted window statistics
# ----------------------------------------------------------------------------------------------
def flat_features(X: np.ndarray) -> np.ndarray:
    """(N, T, F) window -> (N, 9F) statistics: last, mean, std, min, max, slope, mean-of-last-5, 10-step and 20-step change."""
    N, T, F = X.shape
    t = np.arange(T, dtype=np.float32) - (T - 1) / 2
    slope = (X * t[None, :, None]).sum(1) / (t ** 2).sum()
    d10 = X[:, -1] - X[:, max(0, T - 10)]
    d20 = X[:, -1] - X[:, max(0, T - 20)]
    return np.concatenate([X[:, -1], X.mean(1), X.std(1), X.min(1), X.max(1), slope, X[:, -5:].mean(1), d10, d20], axis=1)


def flat_feature_names(names):
    return [f"{n}__{s}" for s in ("last", "mean", "std", "min", "max", "slope", "last5", "d10", "d20") for n in names]


# ----------------------------------------------------------------------------------------------
# 6. Anomaly detection (unsupervised): Mahalanobis distance from the engine's HEALTHY behaviour
# ----------------------------------------------------------------------------------------------
class AnomalyDetector:
    """
    Learns the normal (healthy) sensor distribution from the first part of each training engine's life.
    distance = Mahalanobis distance of the current (smoothed, condition-normalised) sensor vector.
    score in [0,1):  r = distance / alarm-distance ;  score = r^2/(1+r^2)  (0.5 == alarm threshold; the alarm distance is
    first set to the healthy 99.5th percentile and then calibrated on validation engines)
    """

    def __init__(self, healthy_fraction=0.25, quantile=0.995):
        self.healthy_fraction, self.quantile = healthy_fraction, quantile

    def fit(self, healthy_rows: np.ndarray):
        self.mu = healthy_rows.mean(0)
        cov = np.cov(healthy_rows.T) + np.eye(healthy_rows.shape[1]) * 1e-3
        self.prec = np.linalg.inv(cov)
        self.thr = float(np.quantile(self.distance(healthy_rows), self.quantile))
        return self

    def distance(self, rows):
        d = rows - self.mu
        return np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", d, self.prec, d), 0.0))

    def score(self, rows):
        r = self.distance(rows) / max(self.thr, 1e-9)
        return r ** 2 / (1.0 + r ** 2)

    def calibrate(self, rows, labels):
        """Choose the alarm threshold that maximises F1 on VALIDATION engines (label 1 = degraded). Returns (F1, threshold)."""
        d, labels = self.distance(rows), np.asarray(labels).astype(bool)
        best = (-1.0, self.thr)
        for t in np.quantile(d, np.linspace(0.05, 0.98, 150)):
            p = d > t
            tp, fp, fn = int((p & labels).sum()), int((p & ~labels).sum()), int((~p & labels).sum())
            f1 = 2 * tp / max(2 * tp + fp + fn, 1)
            if f1 >= best[0]:
                best = (f1, float(t))
        self.thr = best[1]
        return best

    def to_dict(self):
        return dict(mu=self.mu.tolist(), prec=self.prec.tolist(), thr=self.thr)

    @classmethod
    def from_dict(cls, d):
        o = cls(); o.mu, o.prec, o.thr = np.array(d["mu"]), np.array(d["prec"]), d["thr"]
        return o


def warmup_fix(scores, n=20):
    """The smoothing needs ~n cycles to forget its (noisy) first reading, so early anomaly scores are unreliable:
    cap them at the value reached at cycle n."""
    s = np.array(scores, dtype=float).copy()
    if len(s) > n:
        s[:n] = np.minimum(s[:n], s[n])
    return s


def healthy_rows_from_train(prep: Preprocessor, train_df: pd.DataFrame, fraction=0.25) -> np.ndarray:
    rows = []
    for _, g in train_df.groupby("unit_id"):
        f = prep.sensor_part(prep.transform_engine(g.sort_values("cycle")))
        rows.append(f[: max(5, int(len(f) * fraction))])
    return np.vstack(rows)


# ----------------------------------------------------------------------------------------------
# 7. Health indicators used by the dashboard
# ----------------------------------------------------------------------------------------------
def health_percent(rul, cap=RUL_CAP):
    return float(np.clip(rul / cap, 0, 1) * 100)


def status_from_rul(rul):
    return "HEALTHY" if rul >= 70 else ("WARNING" if rul >= 30 else "CRITICAL")


ACTIONS = {
    "HEALTHY": "Normal operation. Continue routine monitoring.",
    "WARNING": "Schedule maintenance inspection within the next operating window.",
    "CRITICAL": "Immediate maintenance required. Remove from service as soon as possible.",
}


def trend_arrow(series, lookback=15, thr=0.05):
    s = np.asarray(series, float)[-lookback:]
    if len(s) < 3:
        return "→"
    slope = np.polyfit(np.arange(len(s)), s, 1)[0] * lookback
    return "↗" if slope > thr else ("↘" if slope < -thr else "→")


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)))


def load_json(path):
    return json.loads(Path(path).read_text())

In [ ]:
%%writefile src/viz.py
"""viz.py - all figures (matplotlib, one consistent dark industrial theme). Used by the notebooks AND the Streamlit app."""
from __future__ import annotations

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, Wedge

BG, PANEL, GRID, TEXT, MUTED = "#0e1621", "#152231", "#26374a", "#e6edf5", "#8fa3b8"
OK, WARN, CRIT, BLUE, VIOLET = "#2dd4a8", "#f5b83d", "#ef5b5b", "#4ea8ff", "#a78bfa"
STATUS_COLOR = {"HEALTHY": OK, "WARNING": WARN, "CRITICAL": CRIT}
PALETTE = [BLUE, OK, WARN, VIOLET, CRIT, "#7dd3fc", "#fb7185"]


def apply_style():
    plt.rcParams.update({
        "figure.facecolor": BG, "axes.facecolor": PANEL, "savefig.facecolor": BG, "axes.edgecolor": GRID,
        "axes.labelcolor": MUTED, "axes.titlecolor": TEXT, "axes.titleweight": "bold", "axes.titlesize": 12,
        "xtick.color": MUTED, "ytick.color": MUTED, "text.color": TEXT, "grid.color": GRID, "grid.alpha": 0.6,
        "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False, "legend.facecolor": PANEL,
        "legend.edgecolor": GRID, "font.size": 10, "figure.dpi": 100, "savefig.dpi": 150,
    })


apply_style()


def _finish(fig, path=None):
    if path:
        from pathlib import Path
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, bbox_inches="tight")
    return fig


def fig_gauge(value, title, color=None, vmax=100, unit="%", path=None):
    fig, ax = plt.subplots(figsize=(3.2, 2.1))
    ax.axis("off"); ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.2, 1.2); ax.set_aspect("equal")
    frac = float(np.clip(value / vmax, 0, 1))
    color = color or (OK if frac > 0.55 else WARN if frac > 0.25 else CRIT)
    ax.add_patch(Wedge((0, 0), 1, 0, 180, width=0.22, color=GRID))
    ax.add_patch(Wedge((0, 0), 1, 180 - 180 * frac, 180, width=0.22, color=color))
    txt = f"{value:.0f}{unit}" if vmax > 1.5 else f"{value:.2f}"
    ax.text(0, 0.30, txt, ha="center", va="center", fontsize=21, fontweight="bold", color=TEXT)
    ax.text(0, -0.05, title, ha="center", va="center", fontsize=9, color=MUTED)
    return _finish(fig, path)


def fig_status_donut(counts: dict, path=None):
    fig, ax = plt.subplots(figsize=(4, 3.4))
    order = ["HEALTHY", "WARNING", "CRITICAL"]
    vals = [counts.get(k, 0) for k in order]
    ax.pie(vals, colors=[STATUS_COLOR[k] for k in order], startangle=90, counterclock=False, wedgeprops=dict(width=0.38, edgecolor=BG),
           labels=[f"{k}\n{v}" for k, v in zip(order, vals)], textprops=dict(color=TEXT, fontsize=9))
    ax.set_title("Fleet status"); ax.grid(False)
    return _finish(fig, path)


def fig_rul_timeline(cycles, pred, true_capped=None, cap=125, cutoff=None, title="Predicted RUL over the machine's life", path=None):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.axhspan(0, 30, color=CRIT, alpha=0.10); ax.axhspan(30, 70, color=WARN, alpha=0.08); ax.axhspan(70, cap + 5, color=OK, alpha=0.06)
    if true_capped is not None:
        ax.plot(cycles, true_capped, color=MUTED, lw=2, ls="--", label="True RUL (capped)")
    ax.plot(cycles, pred, color=BLUE, lw=2.2, label="Predicted RUL")
    if cutoff is not None:
        ax.axvline(cutoff, color=TEXT, lw=1, ls=":"); ax.text(cutoff, cap, " now", color=TEXT, va="top", fontsize=9)
    ax.set_ylim(0, cap + 5); ax.set_xlabel("Operating cycle"); ax.set_ylabel("RUL (cycles)"); ax.set_title(title); ax.legend(loc="upper right")
    return _finish(fig, path)


def fig_anomaly_timeline(cycles, scores, cutoff=None, title="Anomaly score over time", path=None):
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.fill_between(cycles, scores, color=VIOLET, alpha=0.25); ax.plot(cycles, scores, color=VIOLET, lw=2)
    ax.axhline(0.5, color=CRIT, ls="--", lw=1.2); ax.text(np.min(cycles), 0.53, "alarm threshold", color=CRIT, fontsize=8)
    if cutoff is not None:
        ax.axvline(cutoff, color=TEXT, lw=1, ls=":")
    ax.set_ylim(0, 1.02); ax.set_xlabel("Operating cycle"); ax.set_ylabel("Anomaly score"); ax.set_title(title)
    return _finish(fig, path)


def fig_sensor_trends(cycles, series: dict, cutoff=None, title="Key sensor trends (standardised per operating condition)", path=None):
    fig, ax = plt.subplots(figsize=(9, 3.4))
    for i, (name, y) in enumerate(series.items()):
        ax.plot(cycles, y, lw=1.8, color=PALETTE[i % len(PALETTE)], label=name)
    if cutoff is not None:
        ax.axvline(cutoff, color=TEXT, lw=1, ls=":")
    ax.set_xlabel("Operating cycle"); ax.set_ylabel("z-score"); ax.set_title(title); ax.legend(ncol=max(1, len(series)), loc="upper left", fontsize=8)
    return _finish(fig, path)


def fig_importance(names, values, title="Why this prediction? (sensor contribution)", top=8, path=None):
    names, values = np.asarray(list(names)), np.asarray(list(values), float)
    idx = np.argsort(-np.abs(values))[:top][::-1]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.barh(names[idx], values[idx], color=[CRIT if v > 0 else BLUE for v in values[idx]])
    ax.axvline(0, color=MUTED, lw=1); ax.set_xlabel("contribution to a SHORTER remaining life (red) / longer (blue)"); ax.set_title(title, fontsize=10)
    return _finish(fig, path)


def fig_pred_vs_true(y_true, y_pred, title="Predicted vs actual RUL", path=None):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.3, 1]})
    err = y_pred - y_true
    axs[0].scatter(y_true, y_pred, c=np.where(np.abs(err) <= 10, OK, np.where(np.abs(err) <= 20, WARN, CRIT)), s=26, alpha=0.85, edgecolor="none")
    m = max(y_true.max(), y_pred.max()) + 5
    axs[0].plot([0, m], [0, m], color=MUTED, ls="--", lw=1.2); axs[0].fill_between([0, m], [-10, m - 10], [10, m + 10], color=OK, alpha=0.08)
    axs[0].set_xlabel("True RUL"); axs[0].set_ylabel("Predicted RUL"); axs[0].set_title(title, fontsize=11)
    axs[1].hist(err, bins=24, color=BLUE, alpha=0.9); axs[1].axvline(0, color=TEXT, lw=1)
    axs[1].set_xlabel("error = predicted − true (cycles)"); axs[1].set_title("Error distribution (green band = ±10 cycles)", fontsize=10)
    fig.tight_layout()
    return _finish(fig, path)


def fig_model_comparison(df, metric="Test_RMSE", title=None, lower_better=True, highlight=None, path=None):
    d = df.sort_values(metric, ascending=not lower_better)
    fig, ax = plt.subplots(figsize=(6.2, max(2.8, 0.38 * len(d) + 1)))
    best = highlight or d.iloc[-1]["Model"]
    ax.barh(d["Model"], d[metric], color=[OK if m == best else BLUE for m in d["Model"]])
    for y, v in enumerate(d[metric]):
        ax.text(v, y, f" {v:.3f}" if abs(v) < 5 else f" {v:.1f}", va="center", fontsize=8, color=TEXT)
    if "R2" in metric:
        ax.set_xlim(0, 1.1); ax.axvline(0.75, color=WARN, ls="--", lw=1); ax.text(0.75, len(d) - 0.4, " target 0.75", color=WARN, fontsize=8)
    ax.set_title(title or metric, fontsize=11)
    fig.tight_layout()
    return _finish(fig, path)


def fig_generalization(train_r2, val_r2, test_r2, path=None):
    fig, ax = plt.subplots(figsize=(5, 3))
    labels, vals = ["train", "validation\n(unseen engines)", "official test\n(unseen engines)"], [train_r2, val_r2, test_r2]
    keep = [(l, v) for l, v in zip(labels, vals) if v is not None]
    bars = ax.bar([l for l, _ in keep], [v for _, v in keep], color=[BLUE, VIOLET, OK][: len(keep)], width=0.55)
    for b, (_, v) in zip(bars, keep):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.1); ax.set_ylabel("R²"); ax.set_title("Generalisation check"); ax.axhline(0.75, color=WARN, ls="--", lw=1)
    return _finish(fig, path)


def fig_training_curves(histories: dict, path=None):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for i, (name, h) in enumerate(histories.items()):
        ax.plot(h["loss"], color=PALETTE[i % 7], lw=1.3, alpha=0.7); ax.plot(h["val_loss"], color=PALETTE[i % 7], lw=2, ls="--", label=name)
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("Huber loss  (solid = train, dashed = validation)")
    ax.set_title("Learning curves"); ax.legend(fontsize=8, ncol=2)
    return _finish(fig, path)


def fig_pipeline(path=None):
    fig, ax = plt.subplots(figsize=(13, 3.4)); ax.axis("off"); ax.set_xlim(0, 13); ax.set_ylim(0, 3.4)
    boxes = [("Industrial\nmachine\n(21 sensors)", BLUE), ("Preprocessing\nper-condition z-score\nsmoothing", BLUE), ("Time windows\n30–40 cycles", BLUE),
             ("Deep model\n1D-CNN → BiLSTM\n→ Attention", VIOLET), ("RUL\nprediction", OK), ("Health +\nAnomaly +\nExplanation", WARN), ("Dashboard\n+ API", CRIT)]
    for i, (t, c) in enumerate(boxes):
        x = 0.15 + i * 1.85
        ax.add_patch(FancyBboxPatch((x, 0.8), 1.55, 1.8, boxstyle="round,pad=0.04,rounding_size=0.12", fc=PANEL, ec=c, lw=2))
        ax.text(x + 0.775, 1.7, t, ha="center", va="center", fontsize=8.6, color=TEXT)
        if i < len(boxes) - 1:
            ax.annotate("", (x + 1.85, 1.7), (x + 1.58, 1.7), arrowprops=dict(arrowstyle="-|>", color=MUTED, lw=1.5))
    ax.text(6.5, 3.05, "Predictive-maintenance pipeline: from raw sensor data to explainable machine-health intelligence", ha="center", fontsize=11, fontweight="bold")
    ax.text(6.5, 0.3, "Baselines (Ridge · RandomForest · ExtraTrees · GradientBoosting) and 6 Keras architectures are compared on the same split",
            ha="center", fontsize=9, color=MUTED)
    return _finish(fig, path)

In [ ]:
import sys, os, json, warnings, random
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data" / "raw"
OUT = ROOT / "artifacts" / "phase1"; FIG = OUT / "figures"
for d in (DATA_DIR, FIG): d.mkdir(parents=True, exist_ok=True)
try: display
except NameError: display = print

from src.rul_core import *
from src import viz
np.random.seed(SEED); random.seed(SEED)
ALL_SETS = ["FD001", "FD002", "FD003", "FD004"]
print("Project root:", ROOT)

## 1. Get the dataset
The cell reuses the files if they are already in `data/raw/`; otherwise it tries to download them from NASA. If your network blocks the download, simply drop `CMAPSSData.zip` into `data/raw/` (Colab: Files panel → drag & drop) and run the cell again.

In [ ]:
files = ensure_dataset(DATA_DIR, ALL_SETS)
raw = {c: dict(zip(("train", "test", "rul"), load_subset(files, c))) for c in ALL_SETS}
for c in ALL_SETS:
    print(c, "train", raw[c]["train"].shape, "| test", raw[c]["test"].shape, "| test RUL labels", len(raw[c]["rul"]))
display(raw["FD001"]["train"].head())

## 2. What does each column mean? (data dictionary)
Every row = one engine at one cycle. Besides `unit_id` and `cycle` there are **3 operating settings** and **21 sensors**.
There is **no vibration sensor** in C-MAPSS; fan speed `Nf` and core speed `Nc` are the closest rotating-machinery analogue.

In [ ]:
catalogue = pd.DataFrame([(k, v[0], v[1]) for k, v in SENSOR_INFO.items()], columns=["column", "symbol", "meaning"])
display(catalogue)
catalogue.to_csv(OUT / "sensor_catalogue.csv", index=False)

## 3. The four sub-datasets at a glance

In [ ]:
rows = []
for c, d in raw.items():
    life = d["train"].groupby("unit_id")["cycle"].max()
    rows.append(dict(dataset=c, description=DATASET_CFG[c]["description"], train_engines=d["train"].unit_id.nunique(),
                     test_engines=d["test"].unit_id.nunique(), train_rows=len(d["train"]), test_rows=len(d["test"]),
                     life_min=life.min(), life_median=life.median(), life_max=life.max(),
                     shortest_test_history=d["test"].groupby("unit_id").size().min()))
overview = pd.DataFrame(rows); display(overview); overview.to_csv(OUT / "dataset_overview.csv", index=False)

## 4. Data-quality audit
We never trust data blindly. Checks: missing values, non-finite values, duplicate (engine, cycle) rows, cycle ordering, constant columns,
and whether the number of official test-RUL labels equals the number of test engines.

In [ ]:
audit = []
for c, d in raw.items():
    for split in ("train", "test"):
        f = d[split]; num = f.select_dtypes("number").to_numpy(float)
        const = [col for col in SETTING_COLS + SENSOR_COLS if f[col].nunique() <= 1]
        ordered = all(g["cycle"].is_monotonic_increasing and g["cycle"].iloc[0] == 1 for _, g in f.groupby("unit_id"))
        audit.append(dict(dataset=c, split=split, rows=len(f), engines=f.unit_id.nunique(), missing=int(f.isna().sum().sum()),
                          non_finite=int((~np.isfinite(num)).sum()), duplicate_rows=int(f.duplicated(["unit_id", "cycle"]).sum()),
                          cycles_ordered_from_1=ordered, constant_columns=len(const)))
audit_df = pd.DataFrame(audit); display(audit_df); audit_df.to_csv(OUT / "audit_table.csv", index=False)
labels_ok = {c: len(d["rul"]) == d["test"].unit_id.nunique() for c, d in raw.items()}
print("Test RUL label count matches number of test engines:", labels_ok)
assert audit_df[["missing", "non_finite", "duplicate_rows"]].sum().sum() == 0 and all(labels_ok.values()) and audit_df.cycles_ordered_from_1.all()
print("AUDIT PASSED: no missing / non-finite / duplicate values; cycles ordered; labels consistent.")

In [ ]:
const_by_set = {c: [col for col in SETTING_COLS + SENSOR_COLS if raw[c]["train"][col].nunique() <= 1] for c in ALL_SETS}
for c, cols in const_by_set.items(): print(f"{c}: constant columns = {cols}")
print("\nObservation: in the single-condition sets (FD001/FD003) several sensors never change -> they carry no information and will be dropped.")
print("In the 6-condition sets (FD002/FD004) nothing is constant *raw*, but most sensors only change because the operating condition changes.")

## 5. EDA ① — How long do engines live?
Engines fail after very different numbers of cycles, so we cannot predict RUL from the cycle count alone.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
for ax, c in zip(axs, ALL_SETS):
    life = raw[c]["train"].groupby("unit_id")["cycle"].max()
    ax.hist(life, bins=18, color=viz.BLUE); ax.axvline(life.median(), color=viz.WARN, ls="--"); ax.set_title(f"{c}  (median {life.median():.0f})"); ax.set_xlabel("cycles until failure")
axs[0].set_ylabel("engines"); fig.suptitle("Engine lifetime in the training sets", y=1.04, fontweight="bold")
fig.savefig(FIG / "01_lifetimes.png", bbox_inches="tight"); plt.show()

## 6. EDA ② — Which sensors reveal degradation?
We align engines by *time to failure* (x = 0 is failure). A useful sensor shows a consistent drift as failure approaches.

In [ ]:
tr1 = add_rul(raw["FD001"]["train"])
informative = [s for s in SENSOR_COLS if tr1[s].nunique() > 1]
rho = tr1[informative + ["RUL_raw"]].corr(method="spearman")["RUL_raw"].drop("RUL_raw").sort_values()
top = rho.abs().sort_values(ascending=False).head(4).index.tolist()
fig, axs = plt.subplots(1, 4, figsize=(15, 3.3))
for ax, s in zip(axs, top):
    for u in tr1.unit_id.unique()[:10]:
        g = tr1[tr1.unit_id == u]; ax.plot(-g["RUL_raw"], g[s], lw=1, alpha=0.8)
    ax.set_title(f"{s} ({SENSOR_INFO[s][0]})  ρ={rho[s]:+.2f}", fontsize=10); ax.set_xlabel("cycles before failure")
fig.suptitle("FD001: the four most RUL-correlated sensors (10 engines)", y=1.04, fontweight="bold")
fig.savefig(FIG / "02_sensor_degradation.png", bbox_inches="tight"); plt.show()
print("Sensors with strongest monotonic relation to RUL:", ", ".join(f"{s} ({rho[s]:+.2f})" for s in top))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
colors = [viz.CRIT if abs(v) > 0.5 else viz.BLUE for v in rho.values]
ax.barh(rho.index, rho.values, color=colors); ax.axvline(0, color=viz.MUTED)
ax.set_xlabel("Spearman correlation with RUL (FD001 training data)"); ax.set_title("Sensor relevance (red = |ρ| > 0.5)")
fig.savefig(FIG / "03_sensor_correlation.png", bbox_inches="tight"); plt.show()
print("Sensors that are almost uncorrelated with RUL add noise; the deep model will learn to ignore them, but constant ones are removed outright.")

## 7. EDA ③ — Why FD004 is hard: six operating conditions
In FD002/FD004 the engine is flown at 6 different *operating conditions* (altitude, Mach number, throttle). A sensor then jumps up and down with the
condition — much more than it changes with wear. **If we scale all rows together, the model sees condition noise instead of degradation.**
The fix is to normalise each sensor **within its operating condition**.

In [ ]:
tr4 = add_rul(raw["FD004"]["train"])
p4 = Preprocessor(n_conditions=6, smooth_span=1, use_cycle=False).fit(tr4)
u = tr4.unit_id.iloc[0]; g = tr4[tr4.unit_id == u].sort_values("cycle")
z = p4.transform_engine(g); j = p4.sensors.index("sensor_11")
cond = p4._cond(g)
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
sc = axs[0].scatter(tr4["setting_1"], tr4["setting_2"], c=p4._cond(tr4), cmap="tab10", s=2); axs[0].set_xlabel("setting_1 (altitude)"); axs[0].set_ylabel("setting_2 (Mach)"); axs[0].set_title("6 operating conditions found by k-means")
axs[1].plot(g["cycle"], g["sensor_11"], color=viz.WARN, lw=0.9); axs[1].set_title("RAW sensor_11 of one FD004 engine\n(dominated by condition jumps)"); axs[1].set_xlabel("cycle")
axs[2].plot(g["cycle"], z[:, j], color=viz.OK, lw=0.9); axs[2].set_title("Per-condition NORMALISED sensor_11\n(degradation trend is visible)"); axs[2].set_xlabel("cycle")
fig.savefig(FIG / "04_operating_conditions.png", bbox_inches="tight"); plt.show()
print("Decision: normalise every sensor per operating condition (Phase 2).")

## 8. EDA ④ — The RUL target: why we cap it at 125
An engine that is brand new does not 'look' 300 cycles from failure and 250 cycles from failure differently: degradation only becomes visible late in life. A *piece-wise linear* target (flat at 125, then linearly down to 0) is the standard, physically sensible choice.

In [ ]:
g = tr1[tr1.unit_id == 1]
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(g["cycle"], g["RUL_raw"], color=viz.MUTED, ls="--", label="raw RUL (linear)"); ax.plot(g["cycle"], g["RUL"], color=viz.BLUE, lw=2.5, label=f"capped RUL (cap={RUL_CAP})")
ax.set_xlabel("cycle"); ax.set_ylabel("RUL"); ax.set_title("Piece-wise linear RUL target (engine 1, FD001)"); ax.legend()
fig.savefig(FIG / "05_rul_cap.png", bbox_inches="tight"); plt.show()

## 9. What did we learn? → decisions for Phase 2

In [ ]:
decisions = {
 "constant_columns_FD001": const_by_set["FD001"],
 "top_sensors_FD001": top,
 "operating_conditions": {c: DATASET_CFG[c]["n_conditions"] for c in ALL_SETS},
 "rul_cap": RUL_CAP,
 "decisions": [
   "Split train/validation BY ENGINE (rows of one engine are highly correlated -> row-wise split would leak).",
   "Fit all scalers on training engines only.",
   "Normalise sensors per operating condition (needed for FD002/FD004).",
   "Drop constant sensors; keep the rest (the network decides which matter).",
   "Smooth sensors causally (exponential moving average) to suppress noise without using future data.",
   "Use a piece-wise linear RUL target (cap 125) and evaluate on the official test protocol: LAST window of each test engine."],
}
(OUT / "phase1_findings.json").write_text(json.dumps(decisions, indent=2))
for d in decisions["decisions"]: print("•", d)
print("\nPHASE 1 COMPLETE ✔  — figures in", FIG)